In [1]:
# Ensure we're running from the project root (parent of notebooks/)
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('Working dir:', Path.cwd())

Working dir: /mnt/d/AI/Projects/slonik-7b


# 04 — Error Analysis

What's still failing on Slonik-7B-GRPO at 38.20% BIRD-PG? This notebook categorizes the remaining 309 errors and identifies the realistic paths to higher accuracy.


## Loading the results

In [2]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

with open("results/bird_pg_results_grpo2k.jsonl") as f:
    results = [json.loads(line) for line in f if line.strip()]

stats = Counter(r["status"] for r in results)
total = sum(stats.values())
print(f"Total examples: {total}")
for status, count in stats.most_common():
    pct = count / total * 100
    print(f"  {status:<15} {count:>4} ({pct:>5.1f}%)")

Total examples: 500
  correct          191 ( 38.2%)
  wrong            159 ( 31.8%)
  exec_error       150 ( 30.0%)


## Execution error breakdown

What kinds of errors does Postgres throw on the wrong predictions?


In [3]:
error_types = Counter()
for r in results:
    if r["status"] == "exec_error":
        err = r.get("error", "")
        match = re.match(r"(\w+):", err)
        if match:
            error_types[match.group(1)] += 1

print(f"Total execution errors: {sum(error_types.values())}\n")
print(f"{'Error type':<25} {'Count':>6} {'% of errors':>12}")
print("-" * 50)
total_errors = sum(error_types.values())
for err, count in error_types.most_common(8):
    pct = count / total_errors * 100
    print(f"{err:<25} {count:>6} {pct:>11.1f}%")

Total execution errors: 150

Error type                 Count  % of errors
--------------------------------------------------
UndefinedColumn              109        72.7%
UndefinedFunction             21        14.0%
GroupingError                  7         4.7%
UndefinedTable                 5         3.3%
SyntaxError                    5         3.3%
CardinalityViolation           2         1.3%
InvalidColumnReference         1         0.7%


## Headline insight: schema grounding is still the bottleneck

The vast majority of remaining `exec_error` cases are `UndefinedColumn` — the model is naming columns that don't exist in the schema. This is the 7B context window problem: long DDL strings push schema details out of the model's effective attention.

`UndefinedFunction` is the second class — usually leftover `MONTH()`-style MySQL syntax that GRPO didn't fully eliminate, or PostgreSQL-specific functions the model never learned.


## Wrong-answer breakdown by difficulty

In [4]:
by_diff = defaultdict(Counter)
for r in results:
    by_diff[r["difficulty"]][r["status"]] += 1

print(f"{'Difficulty':<13} {'Correct':>8} {'Wrong':>7} {'ExecErr':>9} {'Total':>7} {'Acc':>7}")
print("-" * 60)
for d in ("simple", "moderate", "challenging"):
    s = by_diff[d]
    total = sum(s.values())
    print(f"{d:<13} {s['correct']:>8} {s['wrong']:>7} {s['exec_error']:>9} {total:>7} {s['correct']/total*100:>6.1f}%")

Difficulty     Correct   Wrong   ExecErr   Total     Acc
------------------------------------------------------------
simple              83      36        29     148   56.1%
moderate            84      80        86     250   33.6%
challenging         24      43        35     102   23.5%


## Observation: hardest tier is hardest in two ways

On challenging queries:
- **More structurally wrong predictions** (`wrong` status) — the SQL runs but returns different results
- **More execution errors** — the model can't even build a valid query against the schema

Easy tier wins are about dialect and basic grounding. Hard tier wins require multi-step reasoning that 7B models can't reliably hold over long contexts.


## What would help

1. **Schema linking** — retrieve only relevant tables/columns and pre-filter the prompt (most realistic v2 improvement)
2. **Longer GRPO** with more `num_generations` (8-16) to reduce stylistic regressions
3. **AST-distance reward** to penalize the over-quoting and unnecessary-DISTINCT patterns
4. **Larger base model** (14B-32B) for the hardest tier, though this defeats the "runs on a laptop" goal

The first three are achievable. The fourth is a different project.


## Sample of the 12 GRPO wins (full SQL diffs)

These are queries where SFT failed and GRPO got it right. They show the qualitative pattern of what execution feedback teaches.


In [5]:
# Load SFT for comparison
with open("results/bird_pg_results.jsonl") as f:
    sft = {json.loads(line)["question_id"]: json.loads(line) for line in f if line.strip()}

grpo = {r["question_id"]: r for r in results}

fixed = [(qid, sft[qid], grpo[qid]) for qid in sft if qid in grpo
         and sft[qid]["status"] != "correct" and grpo[qid]["status"] == "correct"]

print(f"GRPO fixed {len(fixed)} queries SFT had wrong. Showing 3:\n")
for qid, s, g in fixed[:3]:
    print(f"--- Q{qid} [{s['difficulty']}] ---")
    print(f"Question: {s['question'][:150]}")
    print(f"Gold:    {s['gold_sql'][:200]}")
    print(f"SFT:     {s['pred_sql'][:200]}")
    print(f"GRPO:    {g['pred_sql'][:200]}")
    print()

GRPO fixed 34 queries SFT had wrong. Showing 3:

--- Q1533 [moderate] ---
Question: For all the people who paid more than 29.00 per unit of product id No.5. Give their consumption status in the August of 2012.
Gold:    SELECT T2.Consumption FROM transactions_1k AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T1.Price / NULLIF(T1.Amount, 0) > 29.00 AND T1.ProductID = 5 AND T2.Date = '201208'
SFT:     SELECT T3.Segment FROM transactions_1k AS T1 INNER JOIN customers AS T2 ON T1.CustomerID = T2.CustomerID INNER JOIN yearmonth AS T3 ON T2.CustomerID = T3.CustomerID WHERE T1.ProductID = 5 AND T1.Price
GRPO:    SELECT T3.Consumption FROM transactions_1k AS T1 INNER JOIN customers AS T2 ON T1.CustomerID = T2.CustomerID INNER JOIN yearmonth AS T3 ON T2.CustomerID = T3.CustomerID WHERE T1.ProductID = 5 AND T1.P

--- Q1312 [simple] ---
Question: What's Angela Sanders's major?
Gold:    SELECT T2.major_name FROM member AS T1 INNER JOIN major AS T2 ON T1.link_to_major = T2.m